# Training Failure Analysis

Inspect the persisted guardrail surfaces for the baseline training stack. The notebook prefers
real Kedro outputs under `data/`, can target one custom runtime tree through
`MOTIFML_TRAINING_ARTIFACT_ROOT`, and falls back to the tracked fixture bundle and smoke bundle
when runtime artifacts are not available.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from motifml.datasets.tokenized_model_input_runtime_dataset import (
    TokenizedModelInputRuntimeDataset,
)
from motifml.evaluation.structural_checks import StructuralCheckReport
from motifml.evaluation.unknown_tokens import UnknownTokenUsageReport
from motifml.pipelines.tokenization.models import coerce_vocabulary_stats_report
from motifml.training.model_input import TokenizedDocumentRow
from motifml.training.model_input_stats import (
    coerce_model_input_stats_report,
    render_model_input_stats_markdown,
)
from motifml.training.token_codec import (
    coerce_frozen_vocabulary,
    decode_token_ids_to_strings,
)

pd.set_option("display.max_colwidth", 120)


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the MotifML repository root.")


def _load_runtime_rows(model_input_root: Path) -> list[TokenizedDocumentRow]:
    if not model_input_root.exists() or not (model_input_root / "records").exists():
        return []

    runtime_handle = TokenizedModelInputRuntimeDataset(str(model_input_root)).load()
    rows: list[TokenizedDocumentRow] = []
    for split_name in ("train", "validation", "test"):
        rows.extend(
            loaded_document.row
            for loaded_document in runtime_handle.build_document_dataset(
                split=split_name,
                iteration_options={
                    "shuffle_shards": False,
                    "shuffle_documents": False,
                    "shuffle_windows": False,
                },
            )
        )
    return rows


repo_root = _find_repo_root(Path.cwd())
data_root = Path(os.environ.get("MOTIFML_DATA_ROOT", repo_root / "data")).resolve()
training_artifact_root_env = os.environ.get("MOTIFML_TRAINING_ARTIFACT_ROOT")
training_artifact_root = (
    None
    if training_artifact_root_env is None
    else Path(training_artifact_root_env).resolve()
)
fixture_root = Path(
    os.environ.get(
        "MOTIFML_TRAINING_FIXTURE_ROOT",
        repo_root / "tests" / "fixtures" / "training",
    )
).resolve()
reporting_root = (
    training_artifact_root
    if training_artifact_root is not None
    else data_root / "08_reporting" / "training"
)
model_input_root = (
    (training_artifact_root / "model_input")
    if training_artifact_root is not None
    else data_root / "05_model_input" / "ir"
)
smoke_bundle_root = Path(
    os.environ.get(
        "MOTIFML_TRAINING_SMOKE_BUNDLE_ROOT",
        fixture_root / "smoke_bundle",
    )
).resolve()


def load_json(path: Path) -> object:
    return json.loads(path.read_text(encoding="utf-8"))


runtime_paths = {
    "metrics": reporting_root / "metrics.json",
    "vocab_stats": reporting_root / "vocab_stats.json",
    "model_input_stats": reporting_root / "model_input_stats.json",
    "vocabulary": (training_artifact_root / "vocabulary.json")
    if training_artifact_root is not None
    else model_input_root / "vocabulary.json",
}
runtime_rows = _load_runtime_rows(model_input_root.resolve())
if all(path.exists() for path in runtime_paths.values()) and runtime_rows:
    metrics_payload = load_json(runtime_paths["metrics"])
    vocab_stats = coerce_vocabulary_stats_report(
        load_json(runtime_paths["vocab_stats"])
    )
    model_input_stats = coerce_model_input_stats_report(
        load_json(runtime_paths["model_input_stats"])
    )
    vocabulary = coerce_frozen_vocabulary(load_json(runtime_paths["vocabulary"]))
    rows = runtime_rows
    artifact_source = "Runtime Kedro Outputs"
else:
    metrics_payload = load_json(smoke_bundle_root / "metrics.json")
    vocab_stats = coerce_vocabulary_stats_report(
        load_json(fixture_root / "vocab_stats.json")
    )
    model_input_stats = coerce_model_input_stats_report(
        load_json(fixture_root / "model_input_stats.json")
    )
    vocabulary = coerce_frozen_vocabulary(load_json(fixture_root / "vocabulary.json"))
    rows = [
        TokenizedDocumentRow.from_row_dict(load_json(path))
        for path in sorted((fixture_root / "representative_rows").rglob("*.row.json"))
        if path.is_file()
    ]
    artifact_source = "Training Fixtures"

reviewed_split_name = (
    "validation"
    if "validation" in metrics_payload["splits"]
    else next(iter(sorted(metrics_payload["splits"])))
)
reviewed_metrics = metrics_payload["splits"][reviewed_split_name]
split_unknowns = UnknownTokenUsageReport(**reviewed_metrics["unknown_token_usage"])
generated_unknowns = UnknownTokenUsageReport(
    **reviewed_metrics["generated_unknown_token_usage"]
)
structural_report = StructuralCheckReport(**reviewed_metrics["structural"])
row_unknown_summary = []
for row in rows:
    decoded = decode_token_ids_to_strings(row.token_ids, vocabulary=vocabulary)
    unk_count = sum(1 for token in decoded if token == "<unk>")
    row_unknown_summary.append(
        {
            "relative_path": row.relative_path,
            "split": row.split.value,
            "token_count": row.token_count,
            "unk_token_count": unk_count,
            "unk_rate": 0.0 if row.token_count == 0 else unk_count / row.token_count,
        }
    )
row_unknown_summary.sort(key=lambda item: (-item["unk_rate"], item["relative_path"]))
first_failure = structural_report.failures[0] if structural_report.failures else None
worst_document_token_count = (
    model_input_stats.worst_offending_documents[0].token_count
    if model_input_stats.worst_offending_documents
    else 0
)

display(
    Markdown(
        "\n".join(
            (
                "## Failure Mode Overview",
                "",
                f"- Artifact Surface: `{artifact_source}`",
                f"- Reviewed Split: `{reviewed_split_name}`",
                f"- Validation `<unk>` Rate: `{split_unknowns.unk_rate:.6f}` ({split_unknowns.unk_token_count}/{split_unknowns.token_count})",
                f"- Generated `<unk>` Rate: `{generated_unknowns.unk_rate:.6f}` ({generated_unknowns.unk_token_count}/{generated_unknowns.token_count})",
                f"- Vocabulary Guardrails Passed: `{vocab_stats.guardrails.passed}`",
                f"- Structural Failure Count: `{len(structural_report.failures)}`",
                f"- Worst Document Token Count: `{worst_document_token_count}`",
            )
        )
    )
)

In [ ]:
display(
    Markdown(
        "## Unknown Token Review\n\n"
        f"- Highest Representative Row `<unk>` Rate: `{row_unknown_summary[0]['unk_rate']:.6f}`\n"
        f"- Highest Representative Row Document: `{row_unknown_summary[0]['relative_path']}`\n"
        f"- Validation Threshold Passed: `{split_unknowns.passed}`\n"
    )
)
display(pd.DataFrame(row_unknown_summary))

In [ ]:
display(
    Markdown(
        "## Vocabulary Coverage\n\n"
        f"- Vocabulary Size: `{vocab_stats.vocabulary_size}`\n"
        f"- Top Token: `{vocab_stats.guardrails.top_token}`\n"
        f"- Top Token Fraction: `{vocab_stats.guardrails.top_token_fraction:.6f}`\n"
        f"- Missing Required Families: `{', '.join(vocab_stats.guardrails.missing_required_token_families) or 'none'}`\n"
    )
)
display(
    pd.DataFrame(
        [
            {
                "family": entry.family,
                "vocabulary_size": entry.vocabulary_size,
                "token_count": entry.token_count,
            }
            for entry in vocab_stats.token_family_coverage
        ]
    )
)
display(
    pd.DataFrame(
        [
            {"token": entry.token, "count": entry.count}
            for entry in vocab_stats.top_tokens
        ]
    )
)

In [ ]:
failure_rows = [
    {
        "sequence_id": failure.sequence_id,
        "check_name": failure.check_name,
        "token_index": failure.token_index,
        "token": failure.token,
        "message": failure.message,
    }
    for failure in structural_report.failures
]
display(
    Markdown(
        "## Structural Drift\n\n"
        f"- Valid Transition Rate: `{structural_report.valid_transition_rate:.6f}`\n"
        f"- Duration Distribution TV: `{structural_report.duration_distribution_total_variation:.6f}`\n"
        f"- Pitch Out-of-Range Fraction: `{structural_report.out_of_range_pitch_fraction:.6f}`\n"
        f"- First Failure Check: `{first_failure.check_name if first_failure is not None else 'none'}`\n"
        f"- First Failure Document: `{first_failure.sequence_id if first_failure is not None else 'none'}`\n"
    )
)
display(pd.DataFrame(failure_rows))

In [ ]:
display(
    Markdown(
        "## Document Pathologies\n\n"
        + render_model_input_stats_markdown(model_input_stats)
    )
)
display(
    pd.DataFrame(
        [
            {
                "relative_path": entry.relative_path,
                "split": entry.split.value,
                "shard_id": entry.shard_id,
                "token_count": entry.token_count,
                "oversized": entry.exceeds_oversized_threshold,
            }
            for entry in model_input_stats.worst_offending_documents
        ]
    )
)